In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
import os

ZIP_PATH = "/content/drive/My Drive/AIO_Homework/RCNN/data/GARBAGE CLASSIFICATION.zip"
EXTRACT_TO = "/content/data/"

os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_TO)

print("Giải nén xong!")
print(os.listdir(EXTRACT_TO))



Giải nén xong!
['GARBAGE CLASSIFICATION']


In [4]:
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.ops import nms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
DATA_ROOT   = "/content/data/GARBAGE CLASSIFICATION"
CLASS_NAMES = ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
NUM_CLASSES = len(CLASS_NAMES)
# Resize ảnh về kích thước lớn hơn để RPN hoạt động chính xác
TRANSFORM = transforms.Compose([
    transforms.Resize((600, 800)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

In [5]:
class RoIPooling(nn.Module):
    def __init__(self, output_size=7):
        super(RoIPooling, self).__init__()
        self.output_size = output_size
    def forward(self, feature_map, rois, img_size):
        img_H, img_W = img_size
        _, C, Hf, Wf = feature_map.shape
        scale_h = Hf / img_H
        scale_w = Wf / img_W
        pooled_list = []
        for roi in rois:
            x1, y1, x2, y2 = roi
            fx1 = max(0,      min(int(x1 * scale_w), Wf - 1))
            fy1 = max(0,      min(int(y1 * scale_h), Hf - 1))
            fx2 = max(fx1+1,  min(int(x2 * scale_w), Wf))
            fy2 = max(fy1+1,  min(int(y2 * scale_h), Hf))
            roi_feat = feature_map[:, :, fy1:fy2, fx1:fx2]
            pooled = F.adaptive_max_pool2d(roi_feat, self.output_size)
            pooled_list.append(pooled.squeeze(0))
        return torch.stack(pooled_list, dim=0)

In [6]:
def compute_iou_batch(boxes1, boxes2):
    # boxes1: (N, 4), boxes2: (M, 4)
    b1 = boxes1.unsqueeze(1) # (N, 1, 4)
    b2 = boxes2.unsqueeze(0) # (1, M, 4)

    x1 = torch.max(b1[:, :, 0], b2[:, :, 0])
    y1 = torch.max(b1[:, :, 1], b2[:, :, 1])
    x2 = torch.min(b1[:, :, 2], b2[:, :, 2])
    y2 = torch.min(b1[:, :, 3], b2[:, :, 3])

    inter = torch.clamp(x2 - x1, min=0) * torch.clamp(y2 - y1, min=0)
    area1 = (b1[:, :, 2] - b1[:, :, 0]) * (b1[:, :, 3] - b1[:, :, 1])
    area2 = (b2[:, :, 2] - b2[:, :, 0]) * (b2[:, :, 3] - b2[:, :, 1])
    union = area1 + area2 - inter + 1e-6
    return inter / union
def encode_bbox(proposals, gt_boxes):
    Px = (proposals[:, 0] + proposals[:, 2]) / 2
    Py = (proposals[:, 1] + proposals[:, 3]) / 2
    Pw = proposals[:, 2] - proposals[:, 0] + 1e-6
    Ph = proposals[:, 3] - proposals[:, 1] + 1e-6
    Gx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    Gy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
    Gw = gt_boxes[:, 2] - gt_boxes[:, 0] + 1e-6
    Gh = gt_boxes[:, 3] - gt_boxes[:, 1] + 1e-6
    tx = (Gx - Px) / Pw
    ty = (Gy - Py) / Ph
    tw = torch.log(Gw / Pw)
    th = torch.log(Gh / Ph)
    return torch.stack([tx, ty, tw, th], dim=1)
def decode_bbox(proposals, bbox_deltas, cls_ids):
    Px = (proposals[:, 0] + proposals[:, 2]) / 2
    Py = (proposals[:, 1] + proposals[:, 3]) / 2
    Pw = proposals[:, 2] - proposals[:, 0]
    Ph = proposals[:, 3] - proposals[:, 1]
    decoded = []
    for i in range(len(proposals)):
        offset = cls_ids[i].item() * 4
        tx = bbox_deltas[i, offset + 0]
        ty = bbox_deltas[i, offset + 1]
        tw = bbox_deltas[i, offset + 2]
        th = bbox_deltas[i, offset + 3]
        Gx = tx * Pw[i] + Px[i]
        Gy = ty * Ph[i] + Py[i]
        Gw = torch.exp(tw) * Pw[i]
        Gh = torch.exp(th) * Ph[i]
        x1 = Gx - Gw / 2
        y1 = Gy - Gh / 2
        x2 = Gx + Gw / 2
        y2 = Gy + Gh / 2
        decoded.append([x1.item(), y1.item(), x2.item(), y2.item()])
    return torch.tensor(decoded)
def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = max(0,     int((cx - w/2) * img_w))
    y1 = max(0,     int((cy - h/2) * img_h))
    x2 = min(img_w, int((cx + w/2) * img_w))
    y2 = min(img_h, int((cy + h/2) * img_h))
    return x1, y1, x2, y2


In [16]:
class RPN(nn.Module):
    def __init__(self, in_channels=512, mid_channels=512, num_anchors=9):
        super(RPN, self).__init__()
        self.conv = nn.Conv2d(in_channels, mid_channels, 3, padding=1)
        self.cls_head = nn.Conv2d(mid_channels, num_anchors * 2, 1)  # objectness classification
        self.reg_head = nn.Conv2d(mid_channels, num_anchors * 4, 1)  # anchor regression
    def forward(self, x):
        h = F.relu(self.conv(x))
        rpn_cls_scores = self.cls_head(h)
        rpn_bbox_pred = self.reg_head(h)
        rpn_cls_scores = rpn_cls_scores.permute(0, 2, 3, 1).contiguous().view(x.size(0), -1, 2)
        rpn_bbox_pred = rpn_bbox_pred.permute(0, 2, 3, 1).contiguous().view(x.size(0), -1, 4)
        return rpn_cls_scores, rpn_bbox_pred
def generate_anchors(grid_h, grid_w, img_h, img_w, stride=32):
    scales = [128, 256, 512]
    ratios = [0.5, 1.0, 2.0]
    anchors = []
    for y in range(grid_h):
        cy = y * stride + stride / 2.0
        for x in range(grid_w):
            cx = x * stride + stride / 2.0
            for scale in scales:
                for ratio in ratios:
                    h_a = scale * np.sqrt(ratio)
                    w_a = scale / np.sqrt(ratio)
                    anchors.append([cx - w_a/2.0, cy - h_a/2.0, cx + w_a/2.0, cy + h_a/2.0])
    return torch.tensor(anchors, dtype=torch.float32)
def get_proposals_from_rpn(anchors, rpn_cls_scores, rpn_bbox_pred, img_size, is_training=True):
    img_h, img_w = img_size
    probs = F.softmax(rpn_cls_scores[0], dim=1)
    fg_scores = probs[:, 1]
    Px = (anchors[:, 0] + anchors[:, 2]) / 2
    Py = (anchors[:, 1] + anchors[:, 3]) / 2
    Pw = anchors[:, 2] - anchors[:, 0] + 1e-6
    Ph = anchors[:, 3] - anchors[:, 1] + 1e-6
    tx, ty, tw, th = rpn_bbox_pred[0].unbind(dim=1)
    Gx = tx * Pw + Px
    Gy = ty * Ph + Py
    Gw = torch.exp(tw) * Pw
    Gh = torch.exp(th) * Ph
    proposals = torch.stack([
        torch.clamp(Gx - Gw/2, 0, img_w),
        torch.clamp(Gy - Gh/2, 0, img_h),
        torch.clamp(Gx + Gw/2, 0, img_w),
        torch.clamp(Gy + Gh/2, 0, img_h)
    ], dim=1)
    # Filter out small proposals
    w, h = proposals[:, 2] - proposals[:, 0], proposals[:, 3] - proposals[:, 1]
    keep = (w >= 1) & (h >= 1)
    proposals, fg_scores = proposals[keep], fg_scores[keep]

    # If no proposals are left after initial filtering, return empty tensor
    if len(proposals) == 0:
        return torch.tensor([], dtype=torch.float32, device=anchors.device)

    pre_nms_topN = 12000 if is_training else 6000
    post_nms_topN = 2000 if is_training else 1000  # Increased for evaluation debugging

    order = torch.argsort(fg_scores, descending=True)[:pre_nms_topN]
    proposals, fg_scores = proposals[order], fg_scores[order]

    iou_threshold_nms = 0.7 if is_training else 0.3 # Reduced for evaluation debugging
    keep_indices = nms(proposals, fg_scores, iou_threshold=iou_threshold_nms)[:post_nms_topN]
    return proposals[keep_indices]

In [8]:
def get_rpn_targets(anchors, gt_boxes, img_size):
    num_anchors = anchors.size(0)
    labels = torch.empty(num_anchors, dtype=torch.long, device=anchors.device).fill_(-1)
    ious = compute_iou_batch(anchors, gt_boxes)
    max_ious, argmax_ious = ious.max(dim=1)
    gt_max_ious, _ = ious.max(dim=0)
    # Đánh nhãn positive cho anchor có IoU lớn nhất với từng GT box
    for i in range(gt_boxes.size(0)):
        labels[ious[:, i] == gt_max_ious[i]] = 1
    labels[max_ious >= 0.7] = 1
    labels[max_ious < 0.3] = 0
    # Loại bỏ anchor vượt rìa ảnh
    img_h, img_w = img_size
    inside = (anchors[:, 0] >= 0) & (anchors[:, 1] >= 0) & (anchors[:, 2] <= img_w) & (anchors[:, 3] <= img_h)
    labels[~inside] = -1
    # Subsampling về 256 anchors (128 positive, 128 negative)
    pos_idx = torch.where(labels == 1)[0]
    if len(pos_idx) > 128:
        labels[pos_idx[torch.randperm(len(pos_idx))[128:]]] = -1
    neg_idx = torch.where(labels == 0)[0]
    num_neg = 256 - (labels == 1).sum().item()
    if len(neg_idx) > num_neg:
        labels[neg_idx[torch.randperm(len(neg_idx))[num_neg:]]] = -1
    pos_anchor_idx = torch.where(labels == 1)[0]
    bbox_targets = encode_bbox(anchors[pos_anchor_idx], gt_boxes[argmax_ious[pos_anchor_idx]])
    return labels, bbox_targets, pos_anchor_idx
def sample_proposals(proposals, gt_boxes, gt_labels, num_samples=128, pos_ratio=0.25):
    ious = compute_iou_batch(proposals, gt_boxes)
    max_ious, argmax_ious = ious.max(dim=1)
    labels = torch.zeros(proposals.size(0), dtype=torch.long, device=proposals.device)
    matched_gt_boxes = proposals.clone()
    pos_mask = max_ious >= 0.5
    labels[pos_mask] = gt_labels[argmax_ious[pos_mask]]
    matched_gt_boxes[pos_mask] = gt_boxes[argmax_ious[pos_mask]]
    bg_mask = (max_ious < 0.5) & (max_ious >= 0.1)
    labels[bg_mask] = 0
    pos_idx = torch.where(labels > 0)[0]
    num_pos = int(num_samples * pos_ratio)
    if len(pos_idx) > num_pos:
        pos_idx = pos_idx[torch.randperm(len(pos_idx))[:num_pos]]
    neg_idx = torch.where(labels == 0)[0]
    num_neg = num_samples - len(pos_idx)
    if len(neg_idx) > num_neg:
        neg_idx = neg_idx[torch.randperm(len(neg_idx))[:num_neg]]
    keep = torch.cat([pos_idx, neg_idx])
    return proposals[keep], labels[keep], matched_gt_boxes[keep]

In [9]:
class FasterRCNN(nn.Module):
    def __init__(self, num_classes, roi_size=7):
        super(FasterRCNN, self).__init__()
        self.num_classes = num_classes
        self.roi_size = roi_size
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.rpn = RPN(in_channels=512, mid_channels=512, num_anchors=9)
        self.roi_pool = RoIPooling(output_size=roi_size)
        flatten_dim = 512 * roi_size * roi_size
        self.shared_fc = nn.Sequential(
            nn.Linear(flatten_dim, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.cls_head = nn.Linear(4096, num_classes + 1)
        self.bbox_head = nn.Linear(4096, 4 * (num_classes + 1))
    def extract_features(self, img_tensor):
        return self.backbone(img_tensor)
    def forward(self, feature_map, rois, img_size):
        pooled = self.roi_pool(feature_map, rois, img_size)
        pooled = pooled.view(pooled.size(0), -1)
        shared = self.shared_fc(pooled)
        cls_scores = self.cls_head(shared)
        bbox_deltas = self.bbox_head(shared)
        return cls_scores, bbox_deltas

In [10]:
def train_faster_rcnn(model, num_epochs=3, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    img_paths = (sorted(glob.glob(os.path.join(DATA_ROOT, "train/images/*.jpg")))
                 + sorted(glob.glob(os.path.join(DATA_ROOT, "train/images/*.jpeg"))))
    label_dir = os.path.join(DATA_ROOT, "train/labels")
    print(f"Số lượng ảnh: {len(img_paths)}")
    model.train()
    for epoch in range(num_epochs):
        total_rpn_cls, total_rpn_box = 0.0, 0.0
        total_det_cls, total_det_box = 0.0, 0.0
        pbar = tqdm(img_paths, desc=f"Epoch {epoch+1}/{num_epochs}", unit="img")
        for img_path in pbar:
            img_pil = Image.open(img_path).convert("RGB")
            img_w, img_h = img_pil.size
            stem = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(label_dir, stem + ".txt")
            gt_boxes, gt_labels = [], []
            if os.path.exists(label_path):
                with open(label_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5: continue
                        cls_id = int(parts[0]) + 1
                        x1, y1, x2, y2 = yolo_to_xyxy(*map(float, parts[1:5]), img_w, img_h)
                        if x2 > x1 and y2 > y1:
                            gt_boxes.append([x1, y1, x2, y2])
                            gt_labels.append(cls_id)
            if not gt_boxes: continue
            # Chuyển data sang tensor
            img_tensor = TRANSFORM(img_pil).unsqueeze(0).to(device)
            gt_boxes_t = torch.tensor(gt_boxes, dtype=torch.float32, device=device)
            gt_labels_t = torch.tensor(gt_labels, dtype=torch.long, device=device)
            # 1. Forward Backbone & RPN
            feature_map = model.extract_features(img_tensor)
            rpn_cls_scores, rpn_bbox_pred = model.rpn(feature_map)
            # 2. Tạo Anchors & RPN Targets
            Hf, Wf = feature_map.shape[2], feature_map.shape[3]
            anchors = generate_anchors(Hf, Wf, img_h, img_w).to(device)
            rpn_labels, rpn_bbox_targets, pos_anchor_idx = get_rpn_targets(anchors, gt_boxes_t, (img_h, img_w))
            # RPN Loss
            loss_rpn_cls = F.cross_entropy(rpn_cls_scores[0], rpn_labels, ignore_index=-1)
            loss_rpn_box = torch.tensor(0.0, device=device)
            if len(pos_anchor_idx) > 0:
                loss_rpn_box = F.smooth_l1_loss(rpn_bbox_pred[0, pos_anchor_idx], rpn_bbox_targets)
            # 3. Tạo Proposals & Fast R-CNN Targets
            proposals = get_proposals_from_rpn(anchors, rpn_cls_scores, rpn_bbox_pred, (img_h, img_w), is_training=True)
            if len(proposals) == 0: continue
            sampled_props, sampled_labels, sampled_gt_boxes = sample_proposals(proposals, gt_boxes_t, gt_labels_t)
            # 4. Detector Head
            cls_scores, bbox_deltas = model(feature_map, sampled_props.tolist(), (img_h, img_w))
            loss_det_cls = F.cross_entropy(cls_scores, sampled_labels)
            loss_det_box = torch.tensor(0.0, device=device)
            pos_mask = sampled_labels > 0
            if pos_mask.sum() > 0:
                pos_deltas = bbox_deltas[pos_mask]
                pos_lbls = sampled_labels[pos_mask]
                targets = encode_bbox(sampled_props[pos_mask], sampled_gt_boxes[pos_mask])

                batch_idx = torch.arange(pos_lbls.size(0))
                start = pos_lbls * 4
                pred_deltas = torch.stack([pos_deltas[batch_idx, start + k] for k in range(4)], dim=1)
                loss_det_box = F.smooth_l1_loss(pred_deltas, targets)
            # Tổng hợp loss và Backprop
            loss = loss_rpn_cls + loss_rpn_box + loss_det_cls + loss_det_box
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # Logging
            total_rpn_cls += loss_rpn_cls.item()
            total_rpn_box += loss_rpn_box.item()
            total_det_cls += loss_det_cls.item()
            total_det_box += loss_det_box.item()
            pbar.set_postfix({
                "rpn_c": f"{loss_rpn_cls.item():.3f}",
                "rpn_b": f"{loss_rpn_box.item():.3f}",
                "det_c": f"{loss_det_cls.item():.3f}",
                "det_b": f"{loss_det_box.item():.3f}"
            })

        print(f"✅ Epoch {epoch+1} done! rpn_cls={total_rpn_cls/len(img_paths):.4f}, det_cls={total_det_cls/len(img_paths):.4f}\n")
    return model


In [11]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FasterRCNN(num_classes=NUM_CLASSES)

    trained_model = train_faster_rcnn(model, num_epochs=3, lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]


Device: cuda
Số lượng ảnh: 7324


Epoch 1/3: 100%|██████████| 7324/7324 [14:52<00:00,  8.21img/s, rpn_c=0.017, rpn_b=0.000, det_c=0.230, det_b=0.022]


✅ Epoch 1 done! rpn_cls=0.1021, det_cls=0.2130



Epoch 2/3: 100%|██████████| 7324/7324 [04:59<00:00, 24.47img/s]


✅ Epoch 2 done! rpn_cls=0.0000, det_cls=0.0000



Epoch 3/3: 100%|██████████| 7324/7324 [05:17<00:00, 23.10img/s]

✅ Epoch 3 done! rpn_cls=0.0000, det_cls=0.0000



RuntimeError: Parent directory /content/drive/My Drive/RCNN does not exist.

In [13]:
save_path = "/content/drive/My Drive/RCNN/faster_rcnn.pth"
# Tạo thư mục cha nếu nó chưa tồn tại
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(trained_model.state_dict(), save_path)
print(f"Lưu model thành công tại: {save_path}")

Lưu model thành công tại: /content/drive/My Drive/RCNN/faster_rcnn.pth
